# Classical Hopf Layer (S³ → S²) Validation

**Purpose**: Validate all components of `ClassicalHopfLayer` from the `hopf-layers` package.

## Components validated
1. **Hopf map**: S³ → S² with machine-precision S² constraint
2. **Fiber extraction**: atan2-based S¹ phase with clipped gradient STE
3. **Transition detection**: differentiable winding number approximation
4. **Exact reconstruction**: (S², S¹) → S³ inverse Hopf map
5. **SU(2) equivariance**: left multiplication rotates the base point
6. **Gradient flow**: end-to-end differentiability through all components

In [1]:
import math
import torch
import numpy as np

from hopf_layers import (
    ClassicalHopfLayer, HopfOutput,
    hopf_inverse,
    quaternion_normalize, quaternion_multiply,
)

from results_utils import setup_results, save_figure, save_table, save_data

RESULTS = setup_results("01_classical_hopf_validation")

# Device setup for GPU support
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

torch.manual_seed(42)
print(f"hopf-layers imported successfully")
print(f"PyTorch {torch.__version__}")

Results dir: C:\Users\ugure\ccode\categorical-tqft-emergence\hopf-layers\notebooks\results\01_classical_hopf_validation
  figures/  tables/  data/
Device: cuda
hopf-layers imported successfully
PyTorch 2.5.1+cu121


## 1. S² Constraint (Machine Precision)

The Hopf map sends unit quaternions to S²: $x^2 + y^2 + z^2 = 1$.
With exact unit quaternion input, the error should be < 1e-12.

In [2]:
# Gate 1: S² constraint at machine precision
q = torch.randn(64, 4, 16, 16, dtype=torch.float64, device=device)
norm = torch.sqrt((q ** 2).sum(dim=1, keepdim=True))
q = q / norm  # exact unit quaternions

layer = ClassicalHopfLayer(eps=1e-16).to(device)
out = layer(q)

norm_sq = (out.base ** 2).sum(dim=1)  # should be 1.0 everywhere
error = (norm_sq - 1.0).abs().max().item()

print(f"GATE 1: S² constraint")
print(f"  Max |x²+y²+z² - 1| = {error:.2e}")
assert error < 1e-12, f"S² constraint error {error:.2e} exceeds 1e-12"
print(f"  [PASS] (threshold: 1e-12)")

GATE 1: S² constraint


  Max |x²+y²+z² - 1| = 1.11e-15
  [PASS] (threshold: 1e-12)


## 2. Fiber Phase Range

The fiber phase $\phi = \text{atan2}(a_3, a_0)$ should be in $[0, 2\pi)$.

In [3]:
# Gate 2: Fiber range
q_f32 = torch.randn(128, 4, 16, 16, device=device)
layer32 = ClassicalHopfLayer().to(device)
out32 = layer32(q_f32)

fmin = out32.fiber.min().item()
fmax = out32.fiber.max().item()

print(f"GATE 2: Fiber phase range")
print(f"  min(fiber) = {fmin:.6f}")
print(f"  max(fiber) = {fmax:.6f}")
assert fmin >= 0.0, f"Fiber min {fmin} is negative"
assert fmax < 2 * math.pi + 1e-6, f"Fiber max {fmax} exceeds 2*pi"
print(f"  [PASS] (range: [0, 2π))")

GATE 2: Fiber phase range
  min(fiber) = 0.000166
  max(fiber) = 6.283110
  [PASS] (range: [0, 2π))


## 3. Known Fiber Values

For $q = (\cos\phi, 0, 0, \sin\phi)$, the fiber should be $\phi = \text{atan2}(\sin\phi, \cos\phi)$.

In [4]:
# Gate 3: Known fiber values
phi_vals = torch.tensor([0.5, 1.0, 2.0, 3.0, 5.0], device=device)
q_known = torch.zeros(5, 4, 1, 1, device=device)
q_known[:, 0, 0, 0] = torch.cos(phi_vals)
q_known[:, 3, 0, 0] = torch.sin(phi_vals)

out_known = layer32(q_known)
expected = torch.remainder(phi_vals, 2 * math.pi)
extracted = out_known.fiber[:, 0, 0]

print(f"GATE 3: Known fiber values")
for i, phi in enumerate(phi_vals):
    print(f"  phi={phi:.1f}: expected={expected[i]:.4f}, got={extracted[i]:.4f}")

max_err = (extracted - expected).abs().max().item()
assert max_err < 1e-5, f"Known fiber error {max_err:.2e}"
print(f"  Max error: {max_err:.2e}  [PASS]")

GATE 3: Known fiber values
  phi=0.5: expected=0.5000, got=0.5000
  phi=1.0: expected=1.0000, got=1.0000
  phi=2.0: expected=2.0000, got=2.0000
  phi=3.0: expected=3.0000, got=3.0000
  phi=5.0: expected=5.0000, got=5.0000
  Max error: 0.00e+00  [PASS]


## 4. Clipped atan2 STE Demo

Near the singularity (a₀ ≈ 0, a₃ ≈ 0), raw atan2 gradients explode.
Our STE clips them to `[-max_grad, max_grad]`.

In [5]:
# Gate 4: STE clipping near singularity
q_sing = torch.zeros(1, 4, 1, 1, requires_grad=True, device=device)
with torch.no_grad():
    q_sing.data[0, 1, 0, 0] = 1.0    # a1=1 (near singularity for atan2)
    q_sing.data[0, 0, 0, 0] = 1e-10  # a0 ≈ 0
    q_sing.data[0, 3, 0, 0] = 1e-10  # a3 ≈ 0

layer_ste = ClassicalHopfLayer(atan2_max_grad=100.0).to(device)
out_sing = layer_ste(q_sing)
out_sing.fiber.sum().backward()

grad_max = q_sing.grad.abs().max().item()
has_inf = torch.isinf(q_sing.grad).any().item()
has_nan = torch.isnan(q_sing.grad).any().item()

print(f"GATE 4: STE clipping")
print(f"  Max |grad| near singularity: {grad_max:.1f}")
print(f"  Contains inf: {has_inf}")
print(f"  Contains NaN: {has_nan}")
assert not has_inf, "Gradient is infinite"
assert not has_nan, "Gradient is NaN"
assert grad_max < 200, f"Gradient {grad_max} exceeds clip threshold"
print(f"  [PASS] (clipped, not exploding)")

GATE 4: STE clipping
  Max |grad| near singularity: 0.0
  Contains inf: False
  Contains NaN: False
  [PASS] (clipped, not exploding)


## 5. Reconstruction Round-Trip

Decompose → Reconstruct → Re-decompose should recover the same (base, fiber).

In [6]:
# Gate 5: Reconstruction round-trip
q_rt = torch.randn(16, 4, 16, 16, device=device)
out_rt = layer32(q_rt)

# Reconstruct: base is (B,3,Lx,Ly) -> (B,Lx,Ly,3)
base_last = out_rt.base.permute(0, 2, 3, 1)
q_rec = hopf_inverse(base_last, out_rt.fiber)

# Re-decompose
base_check = layer32.hopf_map(q_rec)
fiber_check = layer32.extract_fiber(q_rec)

base_err = (base_check - base_last).abs().max().item()
fiber_err = (fiber_check - out_rt.fiber).abs().max().item()

print(f"GATE 5: Reconstruction round-trip")
print(f"  Max base error:  {base_err:.2e}")
print(f"  Max fiber error: {fiber_err:.2e}")
assert base_err < 1e-3, f"Base error {base_err:.2e}"
assert fiber_err < 1e-3, f"Fiber error {fiber_err:.2e}"
print(f"  [PASS]")

# Also check unit norm
rec_norms = torch.sqrt((q_rec ** 2).sum(dim=-1))
norm_err = (rec_norms - 1.0).abs().max().item()
print(f"  Reconstructed norm error: {norm_err:.2e}")
assert norm_err < 1e-5, f"Norm error {norm_err:.2e}"

GATE 5: Reconstruction round-trip
  Max base error:  4.14e-06
  Max fiber error: 4.77e-07
  [PASS]
  Reconstructed norm error: 1.19e-07


## 6. SU(2) Equivariance

Left-multiplying by identity should not change the base point.

In [7]:
# Gate 6: Equivariance under identity
q_eq = quaternion_normalize(torch.randn(16, 4, device=device))
identity = torch.zeros(16, 4, device=device)
identity[:, 0] = 1.0

base_q = layer32.hopf_map(q_eq)
base_id_q = layer32.hopf_map(quaternion_multiply(identity, q_eq))

eq_err = (base_id_q - base_q).abs().max().item()
print(f"GATE 6: Equivariance (identity)")
print(f"  Max |H(I*q) - H(q)| = {eq_err:.2e}")
assert eq_err < 1e-6, f"Equivariance error {eq_err:.2e}"
print(f"  [PASS]")

GATE 6: Equivariance (identity)
  Max |H(I*q) - H(q)| = 0.00e+00
  [PASS]


## 7. End-to-End Gradient Flow

Gradients should flow through all components: base, fiber, transitions.

In [8]:
# Gate 7: Full gradient flow
q_grad = torch.randn(4, 4, 16, 16, requires_grad=True, device=device)
out_grad = layer32(q_grad)
loss = (out_grad.base.sum() + out_grad.fiber.sum() +
        out_grad.transitions_x.sum() + out_grad.transitions_y.sum())
loss.backward()

has_grad = q_grad.grad is not None
no_nan = not torch.isnan(q_grad.grad).any().item() if has_grad else False
nontrivial = q_grad.grad.abs().sum().item() > 0 if has_grad else False

print(f"GATE 7: End-to-end gradient flow")
print(f"  Has gradient: {has_grad}")
print(f"  No NaN:       {no_nan}")
print(f"  Non-trivial:  {nontrivial}")
assert has_grad and no_nan and nontrivial
print(f"  [PASS]")

GATE 7: End-to-end gradient flow
  Has gradient: True
  No NaN:       True
  Non-trivial:  True
  [PASS]


In [9]:
# Save validation summary
gate_results = {
    "Gate": ["1", "2", "3", "4", "5", "6", "7"],
    "Test": [
        "S2 constraint", "Fiber range", "Known fiber values",
        "STE clipping", "Reconstruction", "Equivariance", "Gradient flow"
    ],
    "Criterion": [
        "|x^2+y^2+z^2-1| < 1e-12",
        "phi in [0, 2pi)",
        "atan2(sin phi, cos phi) = phi",
        "No inf/NaN near singularity",
        "Round-trip error < 1e-3",
        "H(I*q) = H(q)",
        "Non-trivial finite gradients"
    ],
    "Status": ["PASS"] * 7,
}
save_table(
    gate_results,
    "classical_hopf_gates",
    RESULTS,
    caption="Classical Hopf layer validation gates",
    label="tab:classical-gates",
)
save_data(
    {
        "s2_error": [error],
        "fiber_min": [fmin],
        "fiber_max": [fmax],
        "known_fiber_max_err": [max_err],
        "ste_grad_max": [grad_max],
        "reconstruction_base_err": [base_err],
        "reconstruction_fiber_err": [fiber_err],
        "reconstruction_norm_err": [norm_err],
        "equivariance_err": [eq_err],
    },
    "gate_measurements",
    RESULTS,
)
print("All results saved to:", RESULTS)

  Saved: 01_classical_hopf_validation\tables\classical_hopf_gates.csv
  Saved: 01_classical_hopf_validation\tables\classical_hopf_gates.tex
  Saved: 01_classical_hopf_validation\data\gate_measurements.json
  Saved: 01_classical_hopf_validation\data\gate_measurements.csv
All results saved to: C:\Users\ugure\ccode\categorical-tqft-emergence\hopf-layers\notebooks\results\01_classical_hopf_validation


## Summary

| Gate | Test | Criterion | Status |
|------|------|-----------|--------|
| 1 | S² constraint | \|x²+y²+z²-1\| < 1e-12 | PASS |
| 2 | Fiber range | φ ∈ [0, 2π) | PASS |
| 3 | Known fiber values | atan2(sin φ, cos φ) = φ | PASS |
| 4 | STE clipping | No inf/NaN near singularity | PASS |
| 5 | Reconstruction | Round-trip error < 1e-3 | PASS |
| 6 | Equivariance | H(I·q) = H(q) | PASS |
| 7 | Gradient flow | Non-trivial, finite gradients | PASS |